In [ ]:
cd ../..

In [ ]:
import yaml, sys
import pandas as pd
import numpy as np
import plotly.express as px
from src.feature_importance import FeatureImportance
from src.reduce_dimensions import ReduceDimensions
from loguru import logger
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sns.set_context("poster")
sns.set_style("ticks")

np.random.seed(0)

logger.remove()
logger.add(sys.stderr, level="WARNING")

In [ ]:
methods = [
    ("triu", "FR-RSA triu"),
    ("none", "FR-RSA none"),
    ("cholesky", "MLEM cholesky"),
    ("exp", "MLEM exp"),
]
layers = range(1, 13, 4)
epsilons = [1e-2, 5e-3, 1e-3, 5e-4]
all_importances = []
all_spearman = []
with tqdm(total=len(methods) * len(layers) * len(epsilons)) as pbar:
    for param, method in methods:
        for layer in layers:
            for eps in epsilons:
                cfg = f"""
                dataset:
                    path: datasets/relative_clause.csv
                trainer:
                    max_epochs: 1000
                    dataloader_builder:
                        cv: 3
                    representations:
                        model_name: bert-base-uncased
                        layer: {layer}
                    model_builder:
                        param: {param}
                    eps: {eps}
                """
                cfg = yaml.safe_load(cfg)
                fi = FeatureImportance(**cfg)
                i, s, w = fi.compute()
                for e in [i, s]:
                    e["Method"] = method
                    e["Layer"] = layer
                    e["Epsilon"] = f"Epsilon {eps:.0e}"
                all_importances.append(i)
                s["variable"] = "Encoding Spearman"
                all_spearman.append(s.copy())
                s["variable"] = "Training duration (s)"
                s["mean"] = w.training_duration.iloc[0]
                all_spearman.append(s.copy())
                s["variable"] = "Converged?"
                s["mean"] = w.converged.iloc[0]
                all_spearman.append(s.copy())
                s["variable"] = "n_epochs"
                s["mean"] = w.n_epochs.iloc[0]
                all_spearman.append(s)
                pbar.update(1)
all_importances = pd.concat(all_importances)
all_spearman = pd.concat(all_spearman)

In [ ]:
all_importances = all_importances[all_importances.split == "test"]
all_spearman = all_spearman[all_spearman.split == "test"]

In [ ]:
features = (
    all_importances.sort_values("mean", ascending=False)
    .groupby("Method")
    .Feature.apply(lambda x: x.unique()[:5])
    .reset_index()
)
features = features.explode("Feature")

In [ ]:
g = sns.relplot(
    all_importances.merge(features),
    x="Layer",
    y="mean",
    hue="Feature",
    aspect=1.5,
    row="Method",
    col="Epsilon",
    kind="line",
)
g.set_titles(col_template="{col_name}", row_template="{row_name}")

In [ ]:
g = sns.relplot(
    all_spearman,
    x="Layer",
    y="mean",
    aspect=1.5,
    hue="Method",
    col="Epsilon",
    row="variable",
    kind="line",
    facet_kws={"sharey": False},
)
g.set_titles(col_template="{col_name}", row_template="{row_name}")